# Softmax·Temperature·Top-k/p 샘플링 실습 - 추가 실습 코드: Top-K vs Top-P 샘플링

- Tutorial ID: `mod-softmax-temperature-lab`
- Tutorial: Softmax·Temperature·Top-k/p 샘플링 실습
- Section ID: `practice-top-k-top-p`
- Section: 추가 실습 코드: Top-K vs Top-P 샘플링


In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: Top-K vs Top-P 샘플링
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def softmax(logits, temperature=1.0):
    scaled = logits / temperature
    exp_scaled = np.exp(scaled - np.max(scaled))
    return exp_scaled / np.sum(exp_scaled)

def top_k_sampling(probs, k):
    """상위 K개 토큰에서만 샘플링"""
    top_k_indices = np.argsort(probs)[-k:][::-1]
        top_k_probs = probs[top_k_indices]
    # 재정규화
        top_k_probs = top_k_probs / np.sum(top_k_probs)
    return top_k_indices, top_k_probs

def top_p_sampling(probs, p):
    """누적 확률이 p가 될 때까지의 토큰에서 샘플링 (Nucleus)"""
    sorted_indices = np.argsort(probs)[::-1]
        sorted_probs = probs[sorted_indices]
    cumsum = np.cumsum(sorted_probs)
    
    # 누적 확률이 p를 넘는 최소 개수
        mask = cumsum <= p
    # 최소 1개는 포함
    if not mask.any():
                mask[0] = True
    else:
        # p를 넘는 첫 토큰까지 포함
                mask[np.argmax(~mask)] = True
    
    nucleus_indices = sorted_indices[mask]
        nucleus_probs = probs[nucleus_indices]
    # 재정규화
        nucleus_probs = nucleus_probs / np.sum(nucleus_probs)
    return nucleus_indices, nucleus_probs

# 테스트
np.random.seed(42)

# 실제 LLM처럼 많은 토큰 중 일부만 높은 확률
vocab = ["the", "a", "to", "of", "and", "I", "is", "that", "it", "for"]
logits = np.array([3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.3, 0.2, 0.1, 0.05])

probs = softmax(logits)

print("=" * 60)
print("Top-K vs Top-P (Nucleus) 샘플링 비교")
print("=" * 60)

print("\\n원본 확률 분포:")
for token, prob in zip(vocab, probs):
    bar = "█" * int(prob * 40)
    print(f"  {token:6s} {prob:.3f} {bar}")

print("\\n━━━ Top-K (K=3) ━━━")
indices, sampled_probs = top_k_sampling(probs, k=3)
print(f"선택된 토큰: {[vocab[i] for i in indices]}")
print("재정규화된 확률:")
for idx, prob in zip(indices, sampled_probs):
    print(f"  {vocab[idx]:6s} {prob:.3f}")

print("\\n━━━ Top-P (P=0.9) ━━━")
indices, sampled_probs = top_p_sampling(probs, p=0.9)
print(f"선택된 토큰 ({len(indices)}개): {[vocab[i] for i in indices]}")
print("재정규화된 확률:")
for idx, prob in zip(indices, sampled_probs):
    print(f"  {vocab[idx]:6s} {prob:.3f}")

print("\\n📊 비교:")
print("  Top-K: 항상 고정된 K개 토큰에서 샘플링")
print("  Top-P: 확률 분포에 따라 동적으로 토큰 수 결정")
print("         → 분포가 평탄하면 더 많은 토큰 포함")